In [ ]:
!pip install transformers torch pytorch-crf seqeval sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=e2c6ddc46a800d27a47dafee077c2448ab1bde5a9344908ff5c2fbfd77981e4a
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
# --------  IMPORTS --------
import os
import torch
import torch.nn as nn
import random
import pandas as pd
from transformers import AutoTokenizer, CamembertModel
from torchcrf import CRF
from torch.utils.data import DataLoader
from torch.optim import AdamW
from seqeval.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score


# -------- CONFIGURATIONS ET LABELS --------
FICHIER_CONLL = "/content/project-7-at-2026-05-21-19-20-2c60381c.conll"
# Étiquettes exactes converties au format BIO
label_list = [
    "O",
    "B-ALTE", "I-ALTE",
    "B-ALTC", "I-ALTC",
    "B-NONALT", "I-NONALT",
    "B-ALTS", "I-ALTS",
    "B-héméronyme", "I-héméronyme"]

label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for i, label in enumerate(label_list)}

# Initialiser le Tokenizer de CamemBERT
tokenizer = AutoTokenizer.from_pretrained("camembert-base")

# -------- FONCTIONS DE LECTURE ET D'ALIGNEMENT (CHUNKING INTELIGENTE)--------
def lire_conll(chemin_fichier, limite_suave=250, limite_max=400):
    textes_words, textes_labels = [], []
    words, labels = [], []

    with open(chemin_fichier, "r", encoding="utf-8") as f:
        for ligne in f:
            ligne = ligne.strip()
            # Gestion des sauts de ligne
            if not ligne:
                if words:
                    textes_words.append(words)
                    textes_labels.append(labels)
                    words, labels = [], []
                continue
          # Extraction des éléments de la ligne
            elements = ligne.split()
            if len(elements) >= 4:
                mot = elements[0]
                label = elements[3]

                words.append(mot)
                labels.append(label)
                # Logique de chunking intelligent :
                # 1. On vérifie si on a dépassé la limite souple ET si on est sur une ponctuation
                #    Cela permet de couper proprement à la fin d'une phrase.
                atingiu_limite_suave = len(words) >= limite_suave and mot in [".", "?", "!"]
                atingiu_limite_max = len(words) >= limite_max
                # 2. Si on atteint la limite maximale, on coupe par précaution
                if atingiu_limite_suave or atingiu_limite_max:
                    textes_words.append(words)
                    textes_labels.append(labels)
                    words, labels = [], []
# Ajout du dernier bloc si le fichier ne finit pas par une ligne vide
    if words:
        textes_words.append(words)
        textes_labels.append(labels)

    return textes_words, textes_labels

def aligner_tokens_et_labels(mots, labels_phrase):
    """Aligne les labels BIO avec les sous-tokens générés par CamemBERT."""
    # 1. Tokenisation : is_split_into_words=True indique que la liste est déjà découpée en mots
    tokenized_inputs = tokenizer(mots, truncation=True, is_split_into_words=True)
    # word_ids() permet de faire le pont : il donne l'index du mot original pour chaque token
    word_ids = tokenized_inputs.word_ids()

    label_ids = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            # Les tokens spéciaux (<s>, </s>) reçoivent "O"
            label_ids.append(label_to_id["O"])
        elif word_idx != previous_word_idx:
            # Début d'un nouveau mot
            label_ids.append(label_to_id[labels_phrase[word_idx]])
        else:
            # Sous-token du même mot : si c'était B-XXX, ça devient I-XXX
            label = labels_phrase[word_idx]
            if label.startswith("B-"):
                label_ids.append(label_to_id["I-" + label[2:]])
            else: # Sinon on garde le label tel quel (O ou I-)
                label_ids.append(label_to_id[label])

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs


# -------- ARCHITECTURE DU MODÈLE (CAMEMBERT + CRF) --------

class CamembertCRF(nn.Module):
    def __init__(self, num_labels, model_name="camembert-base"):
        super().__init__()
        self.camembert = CamembertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.camembert.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.camembert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        emissions = self.classifier(sequence_output)

        # bool() est important pour que le masque du CRF fonctionne correctement
        mask = attention_mask.bool()

        if labels is not None:
            # Retourne la Perte (Loss)
            loss = -self.crf(emissions, labels, mask=mask, reduction='mean')
            return loss
        else:
            # Retourne la meilleure séquence prédite
            prediction = self.crf.decode(emissions, mask=mask)
            return prediction

# -------- PRÉPARATION DES DONNÉES --------

print(" Chargement et alignement des données...")
phrases_words, phrases_labels = lire_conll(FICHIER_CONLL)

donnees = list(zip(phrases_words, phrases_labels))
random.seed(42)
random.shuffle(donnees)

# Séparation : 80% Entraînement, 20% Test
num_train = int(len(donnees) * 0.8)
train_data = donnees[:num_train]
test_data = donnees[num_train:]

def preparer_dataset(donnees_split):
    return [aligner_tokens_et_labels(w, l) for w, l in donnees_split]

encodings_train = preparer_dataset(train_data)
encodings_test = preparer_dataset(test_data)

def collate_fn(batch):
    """Ajoute du padding pour que les lots (batches) aient la même taille."""
    # On cherche la séquence la plus longue pour aligner les autres dessus.
    max_len = max(len(item['input_ids']) for item in batch)
    input_ids, attention_mask, labels = [], [], []

    for item in batch:
      # Calcul du nombre de tokens à ajouter (padding)
        pad_len = max_len - len(item['input_ids'])
        # - Padding pour les IDs de tokens (en utilisant l'ID du token de padding)
        input_ids.append(item['input_ids'] + [tokenizer.pad_token_id] * pad_len)
        # - Padding pour le masque d'attention (0 signifie "ignorer ce token")
        attention_mask.append(item['attention_mask'] + [0] * pad_len)
        # - Padding pour les labels (on utilise "O" pour signifier "aucun label")
        labels.append(item['labels'] + [label_to_id["O"]] * pad_len)
# Conversion en tenseurs PyTorch
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.long)}

train_loader = DataLoader(encodings_train, batch_size=4, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(encodings_test, batch_size=4, collate_fn=collate_fn)


# -------- INITIALISATION DU MODÈLE ET ENTRAÎNEMENT --------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Appareil utilisé : {device}")

model = CamembertCRF(num_labels=len(label_list)).to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

epochs = 20

print("\nDébut de l'entraînement...")
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        b_input_ids = batch['input_ids'].to(device)
        b_attention_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].to(device)

        loss = model(b_input_ids, b_attention_mask, labels=b_labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Époque {epoch+1}/{epochs} | Perte Moyenne : {total_loss/len(train_loader):.4f}")

# -------- ÉVALUATION, PRÉCISION ET EXPORT EXCEL --------


print("\n Évaluation du modèle sur le jeu de test...")
model.eval()

true_labels_all = []
pred_labels_all = []
tokens_all = []

with torch.no_grad():
    for batch in test_loader:
        b_input_ids = batch['input_ids'].to(device)
        b_attention_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].cpu().numpy()

        # Obtenir les prédictions du modèle
        predictions = model(b_input_ids, b_attention_mask)
        mask = b_attention_mask.cpu().numpy()

        # Filtrer le padding et transformer les IDs en chaînes de caractères
        for i in range(len(predictions)):
            true_seq, pred_seq, token_seq = [], [], []
            input_ids_cpu = b_input_ids[i].cpu().numpy()

            for j in range(len(predictions[i])):
                if mask[i][j] == 1: # Ignorer les zéros du padding

                    token = tokenizer.convert_ids_to_tokens(int(input_ids_cpu[j]))

                    if token not in ["<s>", "</s>"]:
                        token_seq.append(token)
                        true_seq.append(id_to_label[b_labels[i][j]])
                        pred_seq.append(id_to_label[predictions[i][j]])

            true_labels_all.append(true_seq)
            pred_labels_all.append(pred_seq)
            tokens_all.append(token_seq)

print("\n=== RÉSULTATS DÉTAILLÉS ===")
report = classification_report(true_labels_all, pred_labels_all, zero_division=0)
print(report)

print("\n=== MÉTRIQUES GLOBALES ===")
acc_global = accuracy_score(true_labels_all, pred_labels_all)
f1_global = f1_score(true_labels_all, pred_labels_all, zero_division=0)
prec_global = precision_score(true_labels_all, pred_labels_all, zero_division=0)
rec_global = recall_score(true_labels_all, pred_labels_all, zero_division=0)

print(f"Accuracy Globale  : {acc_global:.4f}")
print(f"Précision Globale : {prec_global:.4f}")
print(f"Rappel Global     : {rec_global:.4f}")
print(f"F1-Score Global   : {f1_global:.4f}")


print("\n=== CRÉATION DU FICHIER EXCEL DE COMPARAISON ===")
lignes_excel = []

for phrase_idx, (tokens, trues, preds) in enumerate(zip(tokens_all, true_labels_all, pred_labels_all)):
    for t, true_lab, pred_lab in zip(tokens, trues, preds):

        t_clean = t.replace(' ', '')

        lignes_excel.append({
            "Texte_ID": phrase_idx + 1,
            "Mot/Token": t_clean,
            "Vraie_Etiquette": true_lab,
            "Etiquette_Predite": pred_lab,
            "Correct": "Oui" if true_lab == pred_lab else "Non"})

    lignes_excel.append({
        "Texte_ID": "", "Mot/Token": "", "Vraie_Etiquette": "", "Etiquette_Predite": "", "Correct": ""})

# DataFrame
df = pd.DataFrame(lignes_excel)
nom_fichier = "predictions_vs_realite.xlsx"
df.to_excel(nom_fichier, index=False)

print(f" Fichier généré avec succès : {nom_fichier}")

try:
    from google.colab import files
    files.download(nom_fichier)
except ImportError:
    pass